In [1]:
import importlib.util
import os
from pathlib import Path
import sys

# "prepared" loads existing Parquet; "raw" recomputes the sample in memory.
# Choose the mode before running the notebook from the top.
DATA_SOURCE = "prepared"
if DATA_SOURCE not in {"prepared", "raw"}:
    raise ValueError("DATA_SOURCE must be 'prepared' or 'raw'")

PROJECT_ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook from the project root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Honor an explicitly configured JDK; otherwise use a discovered Java 21 install.
configured_java_home = os.environ.get("JAVA_HOME")
java_candidates = [Path("/usr/lib/jvm/java-21-openjdk-amd64")]
if sys.platform == "win32":
    for java_root in (Path("C:/Program Files/Eclipse Adoptium"), Path("C:/Program Files/Java")):
        java_candidates.extend(sorted(java_root.glob("jdk-21*"), reverse=True))
java_binary = "java.exe" if sys.platform == "win32" else "java"
JAVA_HOME = (
    Path(configured_java_home)
    if configured_java_home
    else next((path for path in java_candidates if (path / "bin" / java_binary).is_file()), None)
)
if JAVA_HOME is None or not (JAVA_HOME / "bin" / java_binary).is_file():
    raise RuntimeError(
        "Configure JAVA_HOME for a compatible JDK (the project uses Java 21). "
        "WSL is recommended when Windows Hadoop native file access is unavailable."
    )
os.environ["JAVA_HOME"] = str(JAVA_HOME)

pyspark_spec = importlib.util.find_spec("pyspark")
if pyspark_spec is None or pyspark_spec.origin is None:
    raise ImportError("PySpark is not installed in this notebook environment")
SPARK_HOME = Path(pyspark_spec.origin).parent
os.environ["SPARK_HOME"] = str(SPARK_HOME)
python_bin = Path(sys.executable).parent
os.environ["PATH"] = os.pathsep.join(
    [str(JAVA_HOME / "bin"), str(SPARK_HOME / "bin"), str(python_bin), os.environ.get("PATH", "")]
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Set heap before the Java gateway launches; restart the kernel to change it.
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 16g pyspark-shell")

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Movies_and_TV.jsonl"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "movies_tv_5m"
ROW_LIMIT = 5_000_000
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

spark = (
    SparkSession.builder
    .appName("amazon-recommender-notebook")
    .master("local[8]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "128")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Release only frames owned by this live session; stale sessions cannot service JVM calls.
for frame_name in ("reviews", "clean_reviews", "interactions", "ranked"):
    cached_frame = globals().pop(frame_name, None)
    if cached_frame is not None and globals().get("_cache_owner") is spark:
        cached_frame.unpersist(blocking=True)
for frame_name in ("train", "validation", "test", "quality", "filtered_quality", "split_counts"):
    globals().pop(frame_name, None)
_cache_owner = spark

print(f"Data source: {DATA_SOURCE}")
print(f"Python: {sys.executable}")
print(f"Java home: {JAVA_HOME}")
print(f"Spark home: {SPARK_HOME}")
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory", "Spark default"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Input path: {RAW_PATH}; exists: {RAW_PATH.exists()}")
print(f"Row limit: {ROW_LIMIT:,}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/03 15:46:35 WARN Utils: Your hostname, DESKTOP-VHOHR7G, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/03 15:46:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/akshay/miniconda3/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/03 15:46:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python: /home/akshay/miniconda3/bin/python
Java home: /usr/lib/jvm/java-21-openjdk-amd64
Spark home: /home/akshay/miniconda3/lib/python3.11/site-packages/pyspark
Spark driver heap: 16g; concurrent tasks: 8
Project root: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject
Input exists: True
Input path: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/raw/Movies_and_TV.jsonl
Row limit: 5,000,000


## Choose one data workflow

Set `DATA_SOURCE` in setup before running from the top. The default `prepared` mode skips raw-data actions and loads the existing four Parquet directories in the final cell. Use `raw` to inspect, clean, filter, and split the sample in memory; the final cell then preserves those newly computed frames. This notebook does not write or replace prepared files.

Saved outputs below are historical results from earlier source code. The corrected raw workflow has not been rerun on the 5M-row dataset. Existing Parquet files also retain the rules used when they were originally created; loading them does not apply current preprocessing fixes. Use a new versioned batch output directory when intentionally regenerating data.

The raw workflow uses the shared functions in `src/data/prepare.py`. Restart the kernel after changing those functions or Spark startup settings.

In [2]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

# A fixed schema avoids scanning the entire JSONL to infer all its fields.
review_schema = "user_id STRING, parent_asin STRING, timestamp BIGINT, rating DOUBLE"

reviews = (
    spark.read.schema(review_schema).json(str(RAW_PATH))
    .select("user_id", F.col("parent_asin").alias("item_id"), "timestamp", "rating")
    .limit(ROW_LIMIT)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Columns:", reviews.columns)
reviews.printSchema()
reviews.show(5, truncate=False)
# The next quality aggregation counts all rows and fills the cache.


Columns: ['user_id', 'item_id', 'timestamp', 'rating']
root
 |-- user_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- rating: double (nullable = true)



[Stage 1:>                                                          (0 + 1) / 1]

+----------------------------+----------+-------------+------+
|user_id                     |item_id   |timestamp    |rating|
+----------------------------+----------+-------------+------+
|AGGZ357AO26RQZVRLGU4D4N52DZQ|B013488XFS|1440385637000|5.0   |
|AGKASBHYZPGTEPO6LWZPVJWB2BVA|B00CB6VTDS|1461100610000|5.0   |
|AG2L7H23R5LLKDKLBEF2Q3L2MVDA|B096Z8Z3R6|1646271834582|3.0   |
|AG2L7H23R5LLKDKLBEF2Q3L2MVDA|B09M14D9FZ|1645937761864|4.0   |
|AG2L7H23R5LLKDKLBEF2Q3L2MVDA|B001H1SVZC|1590639227074|5.0   |
+----------------------------+----------+-------------+------+
only showing top 5 rows


In [3]:
quality = reviews.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_users"),
    F.sum(F.col("item_id").isNull().cast("int")).alias("null_items"),
    F.sum(F.col("timestamp").isNull().cast("int")).alias("null_timestamps"),
    F.sum(F.col("rating").isNull().cast("int")).alias("null_ratings"),
    F.countDistinct("user_id").alias("distinct_users"),
    F.countDistinct("item_id").alias("distinct_items"),
    F.min("rating").alias("minimum_rating"),
    F.max("rating").alias("maximum_rating"),
).collect()[0]

print("Rows:", f"{quality['rows']:,}")
print("Distinct users:", f"{quality['distinct_users']:,}")
print("Distinct items:", f"{quality['distinct_items']:,}")
print("Null counts:", {
    "user_id": quality["null_users"],
    "item_id": quality["null_items"],
    "timestamp": quality["null_timestamps"],
    "rating": quality["null_ratings"],
})
print("Rating range:", quality["minimum_rating"], "to", quality["maximum_rating"])

[Stage 3:>                                                          (0 + 1) / 1]

Rows: 5,000,000
Distinct users: 1,052,914
Distinct items: 452,630
Null counts: {'user_id': 0, 'item_id': 0, 'timestamp': 0, 'rating': 0}
Rating range: 1.0 to 5.0


In [4]:
clean_reviews = (
    reviews
    .where(
        F.col("user_id").isNotNull()
        & F.col("item_id").isNotNull()
        & F.col("timestamp").isNotNull()
    )
    .dropDuplicates(["user_id", "item_id", "timestamp"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)

active_users = (
    clean_reviews.groupBy("user_id").count()
    .where(F.col("count") >= MIN_USER_INTERACTIONS)
    .select("user_id")
)
active_items = (
    clean_reviews.groupBy("item_id").count()
    .where(F.col("count") >= MIN_ITEM_INTERACTIONS)
    .select("item_id")
)

interactions = (
    clean_reviews
    .join(active_users, "user_id", "left_semi")
    .join(active_items, "item_id", "left_semi")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# One action summarizes and materializes the reusable filtered DataFrame.
filtered_quality = interactions.agg(
    F.count("*").alias("rows"),
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("item_id").alias("items"),
).first()
print(f"Rows after filtering: {filtered_quality['rows']:,}")
print(f"Users after filtering: {filtered_quality['users']:,}")
print(f"Items after filtering: {filtered_quality['items']:,}")

# Downstream cells now read cached interactions, so release upstream caches.
clean_reviews.unpersist(blocking=True)
reviews.unpersist(blocking=True)


[Stage 28:==========================================>            (98 + 8) / 128]

Rows after filtering: 3,167,935
Users after filtering: 240,494
Items after filtering: 137,987


DataFrame[user_id: string, item_id: string, timestamp: bigint, rating: double]

In [5]:
from pyspark.sql import Window

user_time_order = Window.partitionBy("user_id").orderBy(
    F.col("timestamp").asc(), F.col("item_id").asc()
)
user_rows = Window.partitionBy("user_id")

# Cache the shared window result so all three splits reuse one sort.
ranked = (
    interactions
    .withColumn("position", F.row_number().over(user_time_order))
    .withColumn("user_interaction_count", F.count("*").over(user_rows))
    .withColumn(
        "split",
        F.when(F.col("position") == F.col("user_interaction_count"), "test")
        .when(
            F.col("position") == F.col("user_interaction_count") - 1, "validation"
        )
        .otherwise("train"),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# One action executes the split and returns just three summary rows.
split_counts = {row["split"]: row["count"]
                for row in ranked.groupBy("split").count().collect()}
for split_name in ("train", "validation", "test"):
    print(f"{split_name.title()} rows: {split_counts.get(split_name, 0):,}")

train = ranked.where(F.col("split") == "train").drop(
    "position", "user_interaction_count", "split"
)
validation = ranked.where(F.col("split") == "validation").drop(
    "position", "user_interaction_count", "split"
)
test = ranked.where(F.col("split") == "test").drop(
    "position", "user_interaction_count", "split"
)
interactions.unpersist(blocking=True)


[Stage 43:===============================================>      (113 + 8) / 128]

Train rows: 2,687,407
Validation rows: 240,034
Test rows: 240,494


DataFrame[item_id: string, user_id: string, timestamp: bigint, rating: double]

In [6]:
# Fast resume: after restarting, run only setup and this cell.
# No raw JSON parsing or temporal split recomputation is needed.
for cached_name in ("reviews", "clean_reviews", "interactions", "ranked"):
    cached_frame = globals().get(cached_name)
    if cached_frame is not None:
        cached_frame.unpersist(blocking=True)

interactions = spark.read.parquet(str(OUTPUT_PATH / "interactions"))
train = spark.read.parquet(str(OUTPUT_PATH / "train"))
validation = spark.read.parquet(str(OUTPUT_PATH / "validation"))
test = spark.read.parquet(str(OUTPUT_PATH / "test"))

print("Loaded prepared Parquet datasets from:")
for split_name in ("interactions", "train", "validation", "test"):
    split_path = OUTPUT_PATH / split_name
    parquet_files = list(split_path.glob("*.parquet"))
    print(f"{split_name}: {len(parquet_files)} Parquet files at {split_path}")


Loaded prepared Parquet datasets from:
interactions: 32 Parquet files at /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/processed/movies_tv_5m/interactions
train: 32 Parquet files at /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/processed/movies_tv_5m/train
validation: 32 Parquet files at /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/processed/movies_tv_5m/validation
test: 32 Parquet files at /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/processed/movies_tv_5m/test
